<a href="https://colab.research.google.com/github/DefexHunter/DefexHunter-ML/blob/CodeMetricsBased-ML/ReadyToBeUsed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

project_path = "/content/drive/MyDrive/DefexHunter/SavedData" #replace with your own path
os.makedirs(project_path, exist_ok=True)

In [3]:
import pickle

with open(f"{project_path}/v1.pkl", "rb") as f:
    v1 = pickle.load(f)

In [4]:
rf_model= v1["rf_model"]
xgb_model= v1["xgb_model"]
svm_model= v1["svm_model"]
best_features= v1["best_features"]
imputer= v1["imputer"]
scaler= v1["scaler"]

In [5]:
import numpy as np

def ensemble_predict(X, rf_model, xgb_model, svm_model, best_features, imputer,
    scaler, weights=(1,1,1)):

    # 1. preprocessing
    X = imputer.transform(X)
    X = scaler.transform(X)

    # 2. feature selection
    X_rf = X[:, best_features["RandomForest"]]
    X_xgb = X[:, best_features["XGBoost"]]
    X_svm = X[:, best_features["SVM"]]

    # 3. model predictions
    rf_prob = rf_model.predict_proba(X_rf)
    xgb_prob = xgb_model.predict_proba(X_xgb)
    svm_prob = svm_model.predict_proba(X_svm)

    # 4. ensemble
    w_rf, w_xgb, w_svm = weights
    final_prob = (w_rf*rf_prob + w_xgb*xgb_prob + w_svm*svm_prob) / sum(weights)

    # 5. final class
    final_pred = np.argmax(final_prob, axis=1)

    return final_pred, final_prob

In [ ]:
#y_val_pred,y_val_pred_prob = ensemble_predict(X_val_scaled, best_rf, best_xgb, best_svm, best_features, imputer, scaler, weights=(2,3,1))